# EDA: cleaned city-daily air quality data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/clean_daily.csv.gz", parse_dates=["date"])
cities = sorted(df["city"].unique())
pivot = df.pivot(index="date", columns="city", values="pm25_mean")

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


def heatmap(matrix, size=(6, 5), cmap="coolwarm", vmin=-1, vmax=1, annotate=True):
    fig, ax = plt.subplots(figsize=size)
    image = ax.imshow(matrix.values, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(matrix.index)), matrix.index)
    ax.grid(False)
    if annotate:
        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                ax.text(j, i, f"{matrix.iloc[i, j]:.2f}", ha="center", va="center")
    fig.colorbar(image)
    plt.show()

## 1. Dataset overview

In [ ]:
print(f"{df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"{df['date'].min().date()} to {df['date'].max().date()}")
display(df.groupby("city")["date"].agg(first="min", last="max", days="count"))
display(df.dtypes.to_frame("dtype"))
display(df.head())

**What it tells:** The size and time span of the data, which cities are present, and that each city has one row per calendar day through the last complete day. Check that the dtypes are numeric before anything else.

## 2. Missing values

In [ ]:
value_columns = ["pm25_mean", "pm25_max", "pm10_mean", "temp_mean", "temp_min", "temp_max", "rh_mean"]
missing = df[value_columns].isna().groupby(df["city"]).mean().mul(100).round(1)
missing.loc["all cities"] = (df[value_columns].isna().mean() * 100).round(1)
display(missing)

**What it tells:** How much of each variable is missing after cleaning, in percent. A high share of missing PM2.5 means that city's series has gaps, which limits any model built on it. Missing temperature or humidity means those columns can't be relied on for that city.

## 3. Data availability by month

In [ ]:
valid = df.dropna(subset=["pm25_mean"])
availability = valid.groupby(["city", pd.Grouper(key="date", freq="MS")]).size().unstack(0).fillna(0).astype(int)
availability.index = availability.index.strftime("%Y-%m")
display(availability)
heatmap(availability.T, size=(11, 3.5), cmap="viridis", vmin=0, vmax=31, annotate=False)

**What it tells:** When the gaps happen. Dark cells are months with few valid days. A city that is thin between October and February has too little data on the smog season, which is the part that matters most. The first and last months may be partial.

## 4. Hours of data per day

In [ ]:
fig, ax = plt.subplots()
ax.hist(df["pm25_hours"], bins=np.arange(0, 26) - 0.5, edgecolor="white")
ax.set_xlabel("hours with a PM2.5 reading in the day")
ax.set_ylabel("city-days")
plt.show()

full_days = (df["pm25_hours"] >= 18).groupby(df["city"]).mean().mul(100).round(1)
display(full_days.to_frame("% of days with at least 18 hours"))

**What it tells:** How complete the days are. Days near 24 hours give reliable daily means. Days near the 18-hour cutoff rest on partial coverage and may be biased toward the hours that happened to be recorded. The spike at 0 is the days with no data at all.

## 5. Summary statistics

In [ ]:
display(df[["pm25_mean", "pm10_mean", "temp_mean", "rh_mean"]].describe().round(1))

pm25_stats = df.groupby("city")["pm25_mean"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(1)
pm25_stats["skew"] = df.groupby("city")["pm25_mean"].skew().round(2)
display(pm25_stats)

**What it tells:** Typical level, spread and extremes per city. A mean well above the median together with positive skew means a few severe days pull the average up, so the median and the 95th percentile describe normal and bad days better than the mean does.

## 6. PM2.5 distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for city in cities:
    axes[0].hist(df.loc[df["city"] == city, "pm25_mean"].dropna(), bins=40, alpha=0.5, density=True, label=city)
axes[0].set_xlabel("daily mean PM2.5 (µg/m³)")
axes[0].set_ylabel("density")
axes[0].legend()
axes[1].boxplot([df.loc[df["city"] == city, "pm25_mean"].dropna() for city in cities])
axes[1].set_xticklabels(cities)
axes[1].set_ylabel("daily mean PM2.5 (µg/m³)")
plt.show()

q1 = df.groupby("city")["pm25_mean"].quantile(0.25)
q3 = df.groupby("city")["pm25_mean"].quantile(0.75)
outlier_limit = q3 + 1.5 * (q3 - q1)
days_above = (df["pm25_mean"] > df["city"].map(outlier_limit)).groupby(df["city"]).sum()
display(pd.DataFrame({"outlier limit": outlier_limit.round(1), "days above": days_above}))

**What it tells:** The shape of daily PM2.5. A right-skewed shape with a long tail of severe days is expected. The boxplot shows which cities are highest in a typical day. Days above the outlier limit are more often real smog episodes than sensor errors, so don't drop them without checking.

## 7. Health categories and WHO guideline

In [ ]:
bins = [0, 9.0, 35.4, 55.4, 125.4, 225.4, np.inf]
labels = ["Good", "Moderate", "Unhealthy for Sensitive Groups", "Unhealthy", "Very Unhealthy", "Hazardous"]
observed = df.dropna(subset=["pm25_mean"])
category = pd.cut(observed["pm25_mean"], bins=bins, labels=labels, include_lowest=True)
share = pd.crosstab(observed["city"], category, normalize="index").mul(100).round(1)
display(share)

ax = share.plot.barh(stacked=True, figsize=(11, 4), colormap="RdYlGn_r")
ax.set_xlabel("% of days")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()

above_who = (observed["pm25_mean"] > 15).groupby(observed["city"]).mean().mul(100).round(1)
display(above_who.to_frame("% of days above the WHO 24-hour guideline (15 µg/m³)"))

**What it tells:** How much of the year each city spends in each US EPA health category, and how often it exceeds the WHO 24-hour guideline. This separates chronic exposure from winter spikes: a city that is mostly orange and red all year has a baseline problem, not just a seasonal one.

## 8. PM2.5 over time

In [ ]:
rolling = pivot.rolling(7, min_periods=4).mean()
fig, axes = plt.subplots(len(cities), 1, figsize=(12, 2.6 * len(cities)), sharex=True, squeeze=False)
for ax, city in zip(axes[:, 0], cities):
    ax.plot(pivot.index, pivot[city], color="lightgray", label="daily")
    ax.plot(rolling.index, rolling[city], label="7-day mean")
    ax.set_ylabel(city)
axes[0, 0].legend(loc="upper right")
axes[-1, 0].set_xlabel("date")
fig.supylabel("PM2.5 (µg/m³)")
plt.show()

**What it tells:** Level, trend and episodes through the year. The 7-day mean smooths daily noise, sharp sustained rises are smog episodes, and breaks in the line are missing data. Compare whether the cities rise and fall together.

## 9. Monthly average

In [ ]:
monthly = df.groupby(["city", pd.Grouper(key="date", freq="MS")])["pm25_mean"].mean().unstack(0)
monthly.index = monthly.index.strftime("%Y-%m")
display(monthly.round(1))
print("highest month:", monthly.idxmax().to_dict())
print("lowest month:", monthly.idxmin().to_dict())

ax = monthly.plot(marker="o", figsize=(12, 5))
ax.set_ylabel("mean PM2.5 (µg/m³)")
plt.xticks(rotation=45)
plt.show()

**What it tells:** The seasonal cycle: which months are worst and best for each city and how steep the change is. The first and last months may be partial, so treat their averages with care.

## 10. Smog season versus the rest of the year

In [ ]:
df["season"] = np.where(df["date"].dt.month.isin([10, 11, 12, 1, 2]), "smog season (Oct-Feb)", "rest of year")
season = df.pivot_table(index="city", columns="season", values="pm25_mean", aggfunc="mean")
season["ratio"] = season["smog season (Oct-Feb)"] / season["rest of year"]
display(season.round(2))

ax = season[["rest of year", "smog season (Oct-Feb)"]].plot.bar(figsize=(10, 5), rot=0)
ax.set_ylabel("mean PM2.5 (µg/m³)")
plt.show()

**What it tells:** The size of the winter effect per city, as the ratio of the October to February mean to the rest of the year. A large ratio means the season alone explains much of the variation, so a model without any seasonal signal will struggle.

## 11. Day-of-week pattern

In [ ]:
weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["weekday"] = df["date"].dt.day_name()
weekday = df.pivot_table(index="weekday", columns="city", values="pm25_mean", aggfunc="mean").reindex(weekdays)
display(weekday.round(1))

ax = weekday.plot.bar(figsize=(11, 5), rot=0)
ax.set_ylabel("mean PM2.5 (µg/m³)")
plt.show()

**What it tells:** Whether weekly activity cycles such as traffic and industry show up. Differences of a few µg/m³ over a single year are probably noise. A consistent weekend drop across cities would be a real effect.

## 12. Correlation between variables

In [ ]:
heatmap(df[["pm25_mean", "pm25_max", "pm10_mean", "temp_mean", "rh_mean"]].corr())

**What it tells:** Linear relationships between pollutants and weather. A strong PM2.5 to PM10 correlation is expected. A negative correlation with temperature mostly reflects the winter smog season, since cold months are also high-emission months, and says little about temperature itself.

## 13. PM2.5 against temperature and humidity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, group in df.groupby("season"):
    axes[0].scatter(group["temp_mean"], group["pm25_mean"], s=8, alpha=0.4, label=name)
    axes[1].scatter(group["rh_mean"], group["pm25_mean"], s=8, alpha=0.4, label=name)
axes[0].set_xlabel("mean temperature (°C)")
axes[1].set_xlabel("mean relative humidity (%)")
for ax in axes:
    ax.set_ylabel("daily mean PM2.5 (µg/m³)")
axes[0].legend()
plt.show()

within_season = {}
for name, group in df.groupby("season"):
    within_season[name] = {
        "temperature": group["pm25_mean"].corr(group["temp_mean"], method="spearman"),
        "humidity": group["pm25_mean"].corr(group["rh_mean"], method="spearman"),
    }
print("Spearman correlation with PM2.5, within each season")
display(pd.DataFrame(within_season).T.round(2))

**What it tells:** Whether the weather relationships hold inside each season or only come from the winter versus summer split. The within-season correlations are the ones that carry information a model can use beyond the season flag.

## 14. PM2.5 to PM10 ratio

In [ ]:
df["pm_ratio"] = df["pm25_mean"] / df["pm10_mean"].where(df["pm10_mean"] > 0)
ratio = df.groupby(["city", pd.Grouper(key="date", freq="MS")])["pm_ratio"].median().unstack(0)
ratio.index = ratio.index.strftime("%Y-%m")
display(df.groupby("city")["pm_ratio"].median().round(2).to_frame("median PM2.5 / PM10"))

ax = ratio.plot(marker="o", figsize=(12, 5))
ax.set_ylabel("median PM2.5 / PM10")
plt.xticks(rotation=45)
plt.show()

**What it tells:** PM2.5 as a share of PM10. A high ratio points to combustion and smoke, which produce fine particles, and a low ratio points to dust. Movement from month to month shows when the source mix changes.

## 15. Correlation between cities

In [ ]:
heatmap(pivot.corr())

**What it tells:** How much the cities move together day to day. High correlation means smog events are regional, so one city's level is informative about another's. Low correlation means local sources dominate.

## 16. Autocorrelation

In [ ]:
lags = range(1, 15)
acf = pd.DataFrame({city: [pivot[city].autocorr(lag) for lag in lags] for city in cities}, index=lags)
display(acf.round(2).head(7))

ax = acf.plot(marker="o", figsize=(10, 5))
ax.set_xlabel("lag (days)")
ax.set_ylabel("autocorrelation")
plt.show()

**What it tells:** How much of today's level carries into later days. A high lag-1 value means yesterday is a strong predictor, and how fast the curve decays shows how many days ahead the series stays predictable.

## 17. Day-to-day change

In [ ]:
change = pivot.diff()
display(change.describe().round(1))
display(change.abs().mean().round(1).to_frame("mean absolute daily change (µg/m³)"))

ax = change.plot.box(figsize=(10, 5))
ax.set_ylabel("day-to-day change in PM2.5 (µg/m³)")
plt.show()

**What it tells:** The typical size and spread of daily swings. The mean absolute change is the error of a forecast that says tomorrow equals today, which is the baseline any model has to beat.

## 18. Worst days

In [ ]:
worst = df.nlargest(10, "pm25_mean")[["date", "city", "pm25_mean", "pm25_max", "pm10_mean", "temp_mean", "rh_mean"]]
display(worst.reset_index(drop=True))

**What it tells:** The ten most polluted city-days with their weather. If they fall in the same days or weeks across cities, they are regional episodes rather than isolated sensor spikes.